In [1]:
from sklearn.preprocessing import (StandardScaler, MinMaxScaler, MaxAbsScaler,
                                   RobustScaler, normalize, Binarizer)

In [3]:
import numpy as np
X = np.array([[1, -1, 2],
              [2, 0, 0],
              [0, 1, -1]], dtype=float)

In [4]:
results = {
    "StandardScaler": StandardScaler().fit_transform(X),
    "MinMaxScaler": MinMaxScaler().fit_transform(X),
    "MaxAbsScaler": MaxAbsScaler().fit_transform(X),
    "RobustScaler": RobustScaler().fit_transform(X),
    "normalize (L2)": normalize(X, norm="l2"),
}

In [5]:
for name, res in results.items():
    print(name)
    print(np.round(res, 3), "\n")

StandardScaler
[[ 0.    -1.225  1.336]
 [ 1.225  0.    -0.267]
 [-1.225  1.225 -1.069]] 

MinMaxScaler
[[0.5   0.    1.   ]
 [1.    0.5   0.333]
 [0.    1.    0.   ]] 

MaxAbsScaler
[[ 0.5 -1.   1. ]
 [ 1.   0.   0. ]
 [ 0.   1.  -0.5]] 

RobustScaler
[[ 0.    -1.     1.333]
 [ 1.     0.     0.   ]
 [-1.     1.    -0.667]] 

normalize (L2)
[[ 0.408 -0.408  0.816]
 [ 1.     0.     0.   ]
 [ 0.     0.707 -0.707]] 



In [8]:
from IPython.display import display, Markdown

display(Markdown("""
**Main idea of each method**

| Method | Idea | Acts on |
|---|---|---|
| StandardScaler | (x - mean) / std, so mean 0 and std 1 | Each feature (column) |
| MinMaxScaler | (x - min) / (max - min), so range [0, 1] | Each feature |
| MaxAbsScaler | x / max(abs(x)), so range [-1, 1], keeps zeros and sign | Each feature |
| RobustScaler | (x - median) / IQR, resistant to outliers | Each feature |
| normalize | Divides each row by its L2 norm, so every row has norm 1 | Each sample (row) |
"""))


**Main idea of each method**

| Method | Idea | Acts on |
|---|---|---|
| StandardScaler | (x - mean) / std, so mean 0 and std 1 | Each feature (column) |
| MinMaxScaler | (x - min) / (max - min), so range [0, 1] | Each feature |
| MaxAbsScaler | x / max(abs(x)), so range [-1, 1], keeps zeros and sign | Each feature |
| RobustScaler | (x - median) / IQR, resistant to outliers | Each feature |
| normalize | Divides each row by its L2 norm, so every row has norm 1 | Each sample (row) |


In [9]:
binarizer = Binarizer(threshold=0.0)
print(binarizer.fit_transform(X))

[[1. 0. 1.]
 [1. 0. 0.]
 [0. 1. 0.]]


**Threshold effect:** Values strictly greater than the threshold become 1; the others become 0.

A higher threshold gives more zeros, while a lower threshold gives more ones. With a threshold of 1, only the value 2 stays 1.

**Discussion:** k-NN and SVM rely on distances, so a feature with a large range can dominate the others. Gradient-based models converge faster and more stably when features have similar scales.

Decision trees split on one feature at a time using thresholds. They are invariant to monotonic rescaling, so scaling is usually not necessary for them.


In [11]:
''' PART 2 '''

' PART 2 '

In [12]:
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder, OneHotEncoder

In [14]:
import pandas as pd
data = pd.DataFrame({
    "Feature 1": ["male", "female", "female", "male"],
    "Feature 2": ["from US", "from Europe", "from Asia", "from Africa"],
    "Feature 3": ["uses Safari", "uses Safari", "uses Firefox", "uses Chrome"],
    "Target": ["US", "France", "Japan", "Algeria"],
})

In [15]:
X_cat = data[["Feature 1", "Feature 2", "Feature 3"]]
y_cat = data["Target"]

In [16]:
ord_enc = OrdinalEncoder()
X_ord = ord_enc.fit_transform(X_cat)
print(X_ord)
print(ord_enc.categories_)

[[1. 3. 2.]
 [0. 2. 2.]
 [0. 1. 1.]
 [1. 0. 0.]]
[array(['female', 'male'], dtype=object), array(['from Africa', 'from Asia', 'from Europe', 'from US'], dtype=object), array(['uses Chrome', 'uses Firefox', 'uses Safari'], dtype=object)]


The numbers are **category codes** assigned in alphabetical order (e.g. female=0, male=1).
They do not represent a ranking: Asia=1 is not "less" than US=3 in any real sense.

In [17]:
label_enc = LabelEncoder()
y_num = label_enc.fit_transform(y_cat)
print(y_num)
print(dict(zip(label_enc.classes_, label_enc.transform(label_enc.classes_))))

[3 1 2 0]
{'Algeria': np.int64(0), 'France': np.int64(1), 'Japan': np.int64(2), 'US': np.int64(3)}


In [18]:
oh = OneHotEncoder(sparse_output=False)
y_onehot = oh.fit_transform(y_cat.to_frame())
print(oh.get_feature_names_out())
print(y_onehot)

['Target_Algeria' 'Target_France' 'Target_Japan' 'Target_US']
[[0. 0. 0. 1.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [1. 0. 0. 0.]]


**Label vs one-hot:** label encoding gives one integer column (Algeria=0, France=1, ...),
which implies an order and distances between classes. One-hot gives one binary column per
class, with exactly one 1 per row, and no order is implied.

**Why one-hot for nominal inputs?** Nominal categories (country, browser) have no natural
order. Arbitrary ordinal codes would make a model believe, for example, that Safari > Firefox
or that US is "three times" Africa. One-hot avoids this false ordering, at the cost of more
columns.

# Conclusion
- **Numerical data:** impute (median/mean), then scale (Standard, MinMax, Robust...) or normalize.
- **Categorical data:** impute (most frequent), then encode (one-hot for nominal, ordinal only if order exists).
- **Text data:** read with the right encoding, clean, then vectorize (e.g. counts / TF-IDF) before modelling.
- Always fit preprocessing on the training set only, to avoid data leakage.